<H1>Extract, combine and format the trial data<H1>

## Balaban et al., 2019

The full dataset can be found and downloaded at: https://osf.io/mzs9e/

1. Import the needed packages.

In [ ]:
import os
import fnmatch
import pandas as pd
import numpy as np

2. Set the directory and the pattern within the file names to be matched.
3. Open the files and compile them into a list called arrays and a concatenated table called final_array.

Choose one of the options below (create the file list here in Python or use the file list created in R). 

In [ ]:
directory = '/Path/to/main_directory/K'
pattern = '*_ChangeDetection_BIGT.txt'

In [ ]:
#OPTION 1!! 

#Create a list with each element corresponding to a unique text file
#This works, but it compiles the files in a different order compared with the corresponding R script. 
#filelist = []

#for root, dirs, files in os.walk(directory):
    #for filename in fnmatch.filter(files, pattern):
        #datafile = os.path.join(root, filename)
        #filelist.append(datafile)

#Concatenate the arrays
#arrays = [np.loadtxt(file, dtype='str') for file in filelist]
#final_array = np.concatenate(arrays, axis=0)

In [ ]:
#Option 2!!

#Alternatively, Make the dataset using the file list that was created in R to check against the R dataset
with open("/Path/to/R_File_List.txt") as file:
    lines = [line.rstrip() for line in file]
    

#Add the paths and save as filelist_R
lines_final = [line.replace("./", "/Path/to/main_directory/K/") for line in lines]

filelist_R = lines_final
print(filelist_R)

#Concatenate the arrays
arrays = [np.loadtxt(file, dtype='str') for file in filelist_R]
final_array = np.concatenate(arrays, axis=0)

In [ ]:
len(arrays) #Should be 3849

In [ ]:
len(final_array) #Should be 462186

4. Create the final dataframe with an index that is unique to each dataset. 

In [ ]:
#Convert the data to a preliminary pandas dataframe

colnames = ("PartID", "set_size", "stim_duration", "change_orig", "response_orig", "rt", "test_location_x", "test_location_y")
data_main = pd.DataFrame(final_array, columns = colnames)
data_main.head(5)

In [ ]:
#Check the data types
data_main.dtypes

In [ ]:
#Check the dimensions
data_main.shape

In [ ]:
#Every column is a string, so try converting to integers 

#data_main['PartID'] = data_main['PartID'].astype('int') #This is where the string error is-the other columns convert to integers
data_main['set_size'] = data_main['set_size'].astype('int')
data_main['stim_duration'] = data_main['stim_duration'].astype('int')
data_main['change_orig'] = data_main['change_orig'].astype('int')
data_main['response_orig'] = data_main['response_orig'].astype('int')
data_main['rt'] = data_main['rt'].astype('int')
data_main['test_location_x'] = data_main['test_location_x'].astype('int')
data_main['test_location_y'] = data_main['test_location_y'].astype('int')
data_main.dtypes

In [ ]:
#Get rid of the PartID column

data_main = data_main.drop(['PartID'], axis = 1)
data_main.head()

In [ ]:
#Create a dataset index column
len(arrays[6][:,1:7]) #Check one dataset per array index

In [ ]:
type(arrays)

In [ ]:
#Outputs an ID column for the final dataframe

def create_index_column(inputlist) :
    # Create the index column
    
    length = [len(array) for array in inputlist]
    index = pd.DataFrame(length).reset_index()
    index.rename(columns={0: 'length'}, inplace=True)
    index['id'] = index.index
    indexcol = index.iloc[index.index.repeat(index['length'])]
    indexcol['trial_num'] = indexcol.index
    indexcol['trial_num'] = indexcol.groupby('index').rank(method='first').astype(int)
    ID = pd.DataFrame(indexcol[['id','trial_num']]).reset_index(drop=True)
       
    return ID

In [ ]:
#Iterate through the list and get the length of each array
lengthlist = []
for array in arrays :
    length = len(array)
    lengthlist.append(length)
    cols = ['length']
    lengths = pd.DataFrame(lengthlist, columns=cols)


In [ ]:
#Create the unique participant ID and row number information in separate data frame
index = pd.DataFrame(lengths).reset_index()
index['id'] = index.index
indexcol = index.iloc[index.index.repeat(index['length'])]
indexcol = pd.DataFrame(indexcol).reset_index()
count_by_id = pd.DataFrame(indexcol[['id', 'length']]).reset_index(drop=True)
count_by_id['trial_num'] = count_by_id.groupby('id').cumcount()
count_by_id = count_by_id[['id', 'trial_num']]

In [ ]:
#Concatenate these to create the final dataframe
data = pd.concat([count_by_id, data_main], axis=1).reset_index(drop=True)
len(data)

In [ ]:
#Number variables starting at 1
data['id'] = data['id']+1
data['trial_num'] = data['trial_num']+1

In [ ]:
#Check the final dataset
data.head()

In [ ]:
#Save as a .csv file
data.to_csv('raw_trial_data.csv')

5. Find the datasets that have > 120 trials and see if the number of trial types is equal across conditions if the extra trials are omitted.

In [ ]:
#Count up the trials in each type (set size x same/diff) for each dataset
trialtype_counts = data.groupby(['id','set_size','change_orig']).count().reset_index()
trialtype_counts

#Filter out the datasets with more than 30 trials per trial type
anom_filter = trialtype_counts['stim_duration'] != 30
filtered_counts = trialtype_counts[anom_filter]

#Write these participants to a csv file to inspect
filtered_counts.to_csv('filtered_counts.csv')

#Write their raw data to a csv file to inspect
filter_IDs = filtered_counts['id'].unique() #There are 54 participants with more than 120 trials
filtered_data = data[data['id'].isin(filter_IDs)]
filtered_data.to_csv('filtered_data.csv')

In [ ]:
#Alternatively
#Count up the trials in each type (set size x same/diff) for each dataset 
#after omitting any greater than 120 trials

#Create the trimmed dataset
data['trial_num'] = data.groupby(['id']).cumcount()
trialnum_filter = data['trial_num'] <= 120
data_analysis = data[trialnum_filter]
len(data_analysis)
data_analysis.to_csv('trimmed_dataset.csv')

#Count up the trials in each type (set size x same/diff) for each dataset
trialtype_counts = data_analysis.groupby(['id','set_size','change_orig']).count().reset_index()
trialtype_counts

#Filter out the datasets with more than 30 trials per trial type
anom_filter = trialtype_counts['stim_duration'] != 30
filtered_counts = trialtype_counts[anom_filter]
filtered_counts
filtered_counts.to_csv('filtered_counts_after_removing_trials_over_120.csv')

#This doesn't result in an even number of trial types per condition for these participants,
#So, use the original dataset

6. Format the dataset.

In [ ]:
#data = data.drop(data.columns[[0]], axis = 1)
data.head()

In [ ]:
data['response'] = data['response_orig'].map({1:0, 2:1, 101:0, 102:1})
data['change'] = data['change_orig'].map({0:0, 5:1})
data.dtypes

In [ ]:
data['change'].isna().sum()

In [ ]:
data['response'].isna().sum()

In [ ]:
data['response_orig'].isna().sum() #The original response variable doesn't have any missing values, so these need to be filled in in the choice variable

In [ ]:
missing = data[data['response'].isna()]
missing

In [ ]:
data['rt'].isna().sum()

In [ ]:
#There are 37 NA obvs in the response variable, so one option is to get rid of these rows.
#data = data[data['response'].notna()]
#data['response'] = data['response'].astype(int)
#data.shape

In [ ]:
#Alternatively, delete these datasets from the dataframe. This is what was used in this study. 
odd_response_ids = data[['id']][data['response'].isna()]
data = data[~data['id'].isin(odd_response_ids['id'])]
data['response'] = data['response'].astype('int')
data.shape

In [ ]:
#Select and order the columns to keep
data_final = data[['id', 'trial_num', 'set_size', 'change', 'response', 'rt', 'stim_duration', 'test_location_x', 'test_location_y']]
data_final.head()

In [ ]:
#Write the processed trial-level dataset to a new csv file
data_final.to_csv('Trial_Data.csv')